<h1 style="font-size: 40px; margin-bottom: 0px;">7.1 ChIP-seq alignment</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

Today, we'll align our datasets to the genome using an aligner called <code>bowtie2</code>. After we perform our alignments, we'll take a look at the resulting files generated by <code>bowtie2</code>, and then we'll load the files into the Integrative Genomics Viewer (IGV) (<a href="https://igv.org/doc/desktop/#DownloadPage/" rel="noopener noreferrer"><u>Download link here</u></a>) to take a look at where reads have aligned to the genome. We'll explore how the reads can be visualized, and how we can begin to see reads piling up on either side of our protein of interest.

While you can follow along in this notebook running command line directly in this notebook, it's preferable to follow along in Terminal.

<strong>Learning objectives:</strong>
<ul>
    <li>Learn to use bowtie2 to perform alignments</li>
    <li>Explore bowtie2 outputs</li>
    <li>Learn to use samtools to work with alignment outputs</li>
    <li>Learn to use IGV to visualize aligned reads</li>
</ul>

<h1 style="font-size: 40px; margin-bottom: 0px;">ChIP-seq alignment with <code>bowtie2</code></h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

Today, we'll be aligning our four truncated datasets to our reference genome in class, and your homework this week will be to align the full datasets. Please note that the alignment for the full dataset will take longer than the class dataset, which is truncated to just 10 million reads.

Since Zanconato et al 2015 aligned their ChIP-seq data to the human genome build <strong>hg19</strong>, we'll be doing the same. We'll be making use of an aligner that Dr. Ingolia talked about in lecture, <code>bowtie2</code>. <a href="https://bowtie-bio.sourceforge.net/bowtie2/index.shtml" rel="noopener noreferrer"><u>Documentation for <code>bowtie2</code> can be found here.</u></a>

<h2>Bowtie2</h2>

Aligning sequences to the genome can be quite tricky and computationally intensive. Recall from lecture that the sequences may not always be a perfect match, and in some cases, there may be short gaps in the sequences to get a good alignment (<strong>Fig 1</strong>). Moreover, the sequence reads that we are trying to align are usually around 50-150bp with millions of read for each sequence file, and each of those reads needs to be aligned to the reference genome, which for humans is approximately 3 billion basepairs long. As Dr. Ingolia mentioned, this presents its own set of issues in terms of the amount of memory needed and also the amount of space the aligner needs to search to find the most likely alignment.

<h4 style="text-align: center;"><strong>Fig 1</strong></h4>
<img src="./images/7_1_fig_1.png" style="height: 250px; margin: auto;"/>
<p style="text-align: center;">Image from: Harvard Chan Bioinformatics Core</p>

<h3>Reference genome and index</h3>

Alignment softwares use a reference genome index to help reduce computational overhead needed for searching through large genomes. While theoretically, you can align using the raw reference genome, converting the reference genome into an organized index enables faster searches through the entire genome to find alignments. <code>bowtie2</code> uses a reference genome index to efficiently and relatively quickly align sequence reads, and to keep it's memory usage on the lower end, <code>bowtie2</code> uses an FM Index (which uses the Burrows-Wheeler Transform mentioned in lecture).

So normally before you perform your alignments, you will need to build an index from your reference genome, which can take some time, especially for larger genome sizes. Fortunately for us, the documentation for <code>bowtie2</code> also provides a link to a pre-built index for hg19. The index files can be found in the shared folder for this course, and we'll use it for our alignments that we'll perform today.

<h3>Alignment</h3>

To get an idea of how to use bowtie2 to align our sequences, let's take a look at how its documentation notes to set it up:

```bash
bowtie2 [options]* -x <bt2-idx> {-1 <m1> -2 <m2> | -U <r> | --interleaved <i> | -b <bam>} [-S <sam>]
```

So to use the bowtie2 aligner, you first specify the <code>bowtie2</code> command followed by options and arguments.

<h3>Prepare directory to receive alignment outputs</h3>

Let's go ahead and prepare a new directory <code>alignment</code> that can receive our output, which will be our alignment files.

In [ ]:
%%bash


<h3>Align sequence data to hg19 reference genome</h3>

Let's align our first sequence file using <code>bowtie2</code>. We'll be using a pre-built index so that we don't have to build the index ourselves in class, which can be a time and memory intensive task depending on the size of the genome. 

For those of you working ahead, our index files for <code>bowtie2</code> alignment can be found under:

<code>~/_shared/bcourses-155788-readonly/chip-seq/chip-index</code>

And all our index files will share the same basename <code>hg19</code>.

We'll work with our truncated dataset in class, which can be found under the following directory path:

<code>~/_shared/bcourses-1557888-readonly/chip-seq/truncated/10M_ctrl_1.fastq</code>

You can use the files directly from the shared folder, and <strong>you don't need to copy the files over.</strong> Since the datasets that we're working with are all large, this will help keep our class's storage footprint down.

In [ ]:
%%bash


<h2>Let's breakdown the code:</h2>

Now let's take a look at the code that we ran to align our <code>10M_ctrl_1.fastq</code> sequence file and break down the input with respect to the information we pulled from the manual.

```console
bowtie2
```

This is the command to call up the aligner.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

```console
-x ~/_shared/bcourses-155788-readonly/chip-seq/chip-index/hg19
```

This argument specifies the file path to where the index files are located along with the basename of the index files.

So for our input, we provided the file path to where the index files were located <code>~/_shared/bcourses-155788-readonly/chip-seq/chip-index/</code> along with the basename of our index files <code>hg19</code>.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

```console
-U ~/_shared/bcourses-1557888-readonly/chip-seq/truncated/10M_ctrl_1.fastq
```

This argument specifies the file that you want to align. <code>bowtie2</code> is able to align paired-end reads indicated as <code>-1 &lt;m1&gt; -2 &lt;m2&gt;</code>, or unpaired reads indicated as<code>-U &lt;r&gt;</code>, interleaved <code>.fastq</code> files indicated as <code>--interleaved &lt;i&gt;</code>, and unaligned BAM files indicated as<code>-b &lt;bam&gt;</code>.

So in our case, the ChIP-seq dataset that we're working with is unpaired, so we can use <code>-U</code> to specify that we're providing unpaired sequence reads followed by the file name for our sequence file.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

```console
-S ~/MCB201B_F2026/Week_07/alignment/10M_ctrl_1.sam
```

Here, we specify our output file including the path to the file. If we want to save the SAM file in a directory that is our current working directory, we don't have to include the file path. Since we'll be saving the output to a directory that is not our current directory, we specify the file path along with the output file name.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

```console
-p 4
```

Lastly, we'll specify a performance option <code>-p</code> and tell <code>bowtie2</code> to run its alignment with parallel search threads in order to increase the speed of the alignment (at the expense of its memory use).

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

<h2>Perform alignment for the remaining <code>.fastq</code> files</h2>

Now that we've done an example for a control file, let's run the alignments for the remaining <code>.fastq</code> files. Once everyone has gotten their alignments running, we'll take a short break before continuing with the next section.

Here, we'll chain together multiple command lines into a single physical line that will run multiple alignments sequentially using a <code>;</code>, and then we'll run the line and take a short break.

In [ ]:
%%bash


<h1 style="font-size: 40px; margin-bottom: 0px;">Exploring <code>bowtie2</code> output</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

<h2>Reading the SAM file</h2>

Recall from lecture that SAM files have a specific structure (<strong>Fig 2</strong>), where an aligned sequence read retains its original identifying information but now this information is also associated with a specific genomic location, and things are just slightly rearranged a bit.

<h4 style="text-align: center;"><strong>Fig 2</strong></h4>
<img src="./images/7_1_fig_2.png" style="height: 300px; margin: auto;"/>
<p style="text-align: center;">Image from Dr. Nick Ingolia</p>

Let's take a look at our <code>10M_ctrl_1.sam</code> output file generated by <code>bowtie2</code>.

For those of you working ahead, you'll make use of the <code>head</code> command and specify that you want to output the first 20 rows of the SAM file, and then pipe the output to the <code>less</code> command to temporarily view it in your Terminal window outside of the stdout.

In [ ]:
%%bash


You should see some header lines indicated by an &commat;. The alignment information are contained in the lines after those header lines.

There are a fair number of columns to the SAM file, so you probably see that a single line will wrap to the next line, making it more difficult to read. You might have noticed that the values within the SAM file are separated by tabs, so try to export just the first 100 lines of our SAM file into a tab-separated values <code>.tsv</code> file. That way, we can open it up just the first 100 lines in Excel to see each column more easily.

For those of you working ahead, instead of piping the output to <code>less</code>, redirect your output to a new file. Once we've exported the first 100 lines into a <code>.tsv</code> file, let's open it up in Excel to view its contents.

In [ ]:
%%bash


Unfortunately, the columns do not come with headers, so <a href="https://bowtie-bio.sourceforge.net/bowtie2/manual.shtml#sam-output" rel="noopener noreferrer"><u>we'll have to look into the <code>bowtie2</code> documentation to figure out what each column indicates.</u></a>

To figure out what the FLAG column is telling us, we can look back to the bowtie2 documentation, or <a href="https://broadinstitute.github.io/picard/explain-flags.html" rel="noopener noreferrer"><u>we can also use the Broad Institute's sam FLAG  decoder.</u></a>

For the optional tags, their definitions can be found also in the <code>bowtie2</code> documentation.

<h2>Sort and index alignments</h2>

Recall from lecture that genome viewers usually assume that the alignment data that you give it is sorted by its mapped location. The genome viewer we will be using (IGV) requires us to provide it with a sorted and indexed BAM alignment file in order for it to display the aligned reads.

<h3>Convert SAM to BAM</h3>

The first thing we'll need to do is to convert our SAM file to a BAM file, which is a compressed, binary file. To do this, we'll use the <code>samtools</code> command. <a href="https://www.htslib.org/doc/samtools.html" rel="noopener noreferrer"><u>Documentation for <code>samtools</code> can be found here.</u></a> And <a href="https://www.htslib.org/doc/samtools-view.html" rel="noopener noreferrer"><u>documentation for its <code>view</code> subcommand can be found here.</u></a>

In [ ]:
%%bash


<h2>Let's breakdown the code:</h2>

```console
samtools
```

This is the command to call up <code>samtools</code>.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

```console
view
```

This indicates that we want to make use of the sub-command <code>view</code>, which is used to convert SAM to BAM and vice versa.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

```console
-b
``` 

This tells the view sub-command that we want it to output a BAM file, so essentially convert the SAM file to a BAM file.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

```console
-@ 3
```

Like <code>bowtie2</code>, <code>samtools</code> also has its own performance option. In this case, the performance option is specified by <code>-@</code>. And we'll specify the number of additional parallel threads to process our SAM file into a BAM file. This will help reduce the time needed for the SAM file to be compressed into a BAM file by making use of our server's architecture.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

```console
-o 10M_ctrl_1.bam
``` 

This specifies the output file information. Here is where we provide the sub-command with the file name.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

```console
10M_ctrl_1.sam
``` 

This is the name of the file that we are providing to the sub-command to convert to a BAM file.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

<h3>Sort the BAM file</h3>

To sort your BAM file, you can use the <code>sort</code> sub-command within the <code>samtools</code> command. <a href="https://www.htslib.org/doc/samtools-sort.html" rel="noopener noreferrer"><u>Documentation for the <code>sort</code> subcommand can be found here.</u></a>

In [ ]:
%%bash


<h2>Let's breakdown the code:</h2>

```console
samtools
``` 

This is the command to call up <code>samtools</code> just like before.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

```console
sort
```

This calls up the <code>sort</code> sub-command, which gives us the ability to sort our aligned reads by their chromosomal position.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

```console
-@ 4
```

This is the performance option like we've seen for the <code>view</code> subcommand, and it will allow us to specify parallel threads to sort and compress our BAM file.


<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

```console
-o 10M_ctrl_1-sorted.bam
``` 

This specifies the output file information.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

```console
10M_ctrl_1.bam
``` 

This is the file that contains the alignments that we want to sort based on their chromosomal position.

<hr style="border: 1px solid; border-color: #BBBBBB;"></hr>

<h3>Index the sorted BAM file</h3>

Like with how we needed to create a genome index to allow <code>bowtie2</code> to quickly access the sequence information, we need to index our sorted BAM file to allow for quick access to our alignment data. To index our sorted BAM files, we will use the <code>index</code> sub-command and provide it with the BAM file we want it to index:

In [ ]:
%%bash


You should now have three outputs from converting, sorting, and indexing:

<ul>
    <li><code>10M_ctrl_1.bam</code></li>
    <li><code>10M_ctrl_1-sorted.bam</code></li>
    <li><code>10M_ctrl_1-sorted.bai</code></li>
</ul>

<h3>Delete the SAM file</h3>

What you'll see is that your SAM files are quite large in size, whereas your BAM files are much smaller. This is because your BAM files are compressed, and your SAM files are not. Since the BAM files contain the same information as our original SAM files, we can go ahead and delete the SAM files to keep our storage footprint minimal. We can always uncompress our BAM file if we ever need the SAM file again.

In [ ]:
%%bash


<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise: Convert SAM to BAM, then sort & index</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

Now that we've done one example file together, go ahead and finishing processing your remaining SAM files. Don't forget to delete your SAM files once you're done to save space.

In [ ]:
%%bash


<h1 style="font-size: 40px; margin-bottom: 0px;">Visualize aligned reads</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

To prepare to visualize your aligned reads in IGV, download the following files:

<ol>
    <li>Your sorted BAM files for: <code>10M_ctrl_1-sorted.bam</code> and <code>10M_taz_1-sorted.bam</code></li>
    <li>The sorted and indexed BAM files for: <code>10M_ctrl_1-sorted.bai</code> and <code>10M_taz_1-sorted.bai</code></li>
</ol>

Now, let's open up IGV and visualize our alignments.

<h1 style="font-size: 40px; margin-bottom: 0px;">References</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

<p style="padding-left: 20px;"><a href="https://www.nature.com/articles/nmeth.1923" rel="noopener noreferrer" target="_blank"><u>Langmead and Salzberg Nature Methods</u></a> Fast gapped-read alignment with Bowtie 2</p>

<p style="padding-left: 20px;"><a href="https://academic.oup.com/bioinformatics/article/25/16/2078/204688" rel="noopener noreferrer" target="_blank"><u>Li et al 2009 Bioinformatics</u></a> The Sequence Alignment/Map format and SAMtools</p>

<p style="padding-left: 20px;"><a href="https://www.nature.com/articles/nbt.1754" rel="noopener noreferrer" target="_blank"><u>Robinson et al Nature Biotechnology</u></a> Integrative Genomics Viewer</p>

<p style="padding-left: 20px;"><a href="https://www.nature.com/articles/ncb3216" rel="noopener noreferrer" target="_blank"><u>Zanconato et al 2015 Nat Cell Biol</u></a> ChIP–seq: advantages and challenges of a maturing technology</p>